# Surya Table Extraction Demo

This notebook runs a full table-structure extraction evaluation pipeline:
- load dataset samples
- run Surya inference
- evaluate with `torchmetrics` mAP/mAR
- save metrics and predictions for review.

In [1]:
from __future__ import annotations

## Environment Setup

Install dependencies for this notebook runtime (`uv`, project deps, `torchmetrics`, and compatibility-pinned `transformers`).

In [2]:
toml_content = """
[project]
name = "stp26-table-extraction-machathon-7"
version = "0.1.0"
description = "Add your description here"
requires-python = ">=3.12"
dependencies = [
    "datasets>=4.5.0",
    "evaluate>=0.4.6",
    "ipykernel>=7.2.0",
    "pandas>=3.0.1",
    "surya-ocr>=0.17.1",
    "torchmetrics>=1.8.2",
    "transformers==4.56.1",
]
"""

with open("pyproject.toml", "w") as f:
    f.write(toml_content)

In [3]:
!pip install uv
!uv pip install --system -r pyproject.toml
!uv pip install transformers==4.56.1

'pip' is not recognized as an internal or external command,
operable program or batch file.


^C


Using Python 3.14.0 environment at: C:\Users\yousi\AppData\Local\Programs\Python\Python314
Resolved 92 packages in 1.09s


^C


Audited 1 package in 72ms


## Imports

Imports are grouped by purpose: core Python utilities, data processing, image IO, TorchMetrics, and progress/reporting tools.

In [41]:
import json
import math
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageDraw
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from tqdm.auto import tqdm

## Configuration

Tune runtime behavior from `EvalConfig` (dataset choice, model options, inference chunking, output paths, and metric thresholds).

In [42]:
@dataclass
class EvalConfig:
    # Dataset settings
    dataset_name: str = "ysif9/test-dataset"
    split: str = "train"

    # Sampling / runtime
    max_samples: Optional[int] = None
    random_seed: int = 42

    # Surya inference options
    device: Optional[str] = None
    use_layout_table_detection: bool = True
    table_rec_batch_size: Optional[int] = None
    layout_batch_size: Optional[int] = None
    include_table_of_contents_as_table: bool = False
    fallback_to_full_image_when_no_table: bool = True

    # Inference chunking
    inference_chunk_size: int = 100
    max_table_crops_per_pass: int = 128
    clear_cuda_cache_between_chunks: bool = True
    release_images_after_inference: bool = False

    # Table crop preprocessing
    table_crop_expand_ratio: float = 0.01

    # Header / class mapping heuristics
    use_cell_based_header_mapping: bool = True
    header_row_cell_fraction_threshold: float = 0.45
    header_row_min_width_fraction: float = 0.75
    max_top_header_rows: int = 2
    emit_column_header_band_union: bool = True

    map_projected_row_header_from_non_top_header_rows: bool = True
    enable_projected_row_header_geom_fallback: bool = True
    projected_row_header_min_width_fraction: float = 0.85
    projected_row_header_max_cells: int = 2

    # Spanning-cell heuristics
    enable_span_geometric_fallback: bool = True
    span_relaxed_row_overlap_ratio_threshold: float = 0.10
    span_relaxed_col_overlap_ratio_threshold: float = 0.15
    span_min_row_height_ratio: float = 1.15
    span_min_col_width_ratio: float = 1.15
    enable_unmerged_vertical_span_recovery: bool = True
    vertical_span_min_x_overlap_ratio: float = 0.65
    vertical_span_max_row_gap: int = 2

    # Structural span policy
    use_merged_structural_spans: bool = True
    use_unmerged_structural_spans: bool = True
    structural_span_max_rowspan: int = 100
    structural_span_max_colspan: int = 100
    span_max_area_fraction_of_table: float = 0.35
    span_dedup_iou: float = 0.88
    span_grid_score_alpha: float = 0.05
    span_use_grid_rebuilt_boxes: bool = True

    # Prediction cleanup
    dedupe_predictions: bool = True
    dedupe_iou_threshold: float = 0.98

    # Evaluation
    map_thresholds: Tuple[float, ...] = (
        0.50,
        0.55,
        0.60,
        0.65,
        0.70,
        0.75,
        0.80,
        0.85,
        0.90,
        0.95,
    )

    # Output
    output_dir: str = "outputs/surya_table_eval_full"
    save_visualizations: bool = False
    num_visualizations: int = 20


CONFIG = EvalConfig()

## Data Structures

Core dataclasses:
- `Sample` for GT annotations per image
- `Prediction` for model outputs per image.

In [43]:
CLASS_NAMES: List[str] = [
    "table",
    "table column",
    "table row",
    "table column header",
    "table projected row header",
    "table spanning cell",
]
CLASS_NAME_TO_ID = {name: idx for idx, name in enumerate(CLASS_NAMES)}


@dataclass
class Sample:
    image_id: int
    file_name: str
    image: Image.Image
    gt_boxes_xyxy: List[List[float]]
    gt_labels: List[int]


@dataclass
class Prediction:
    image_id: int
    bbox_xyxy: List[float]
    category_id: int
    score: float
    source: str = ""


## Geometry Utilities

Low-level box helpers (format conversion, clipping, overlap/IoU, shifting) used throughout inference and post-processing.

In [44]:
def coco_xywh_to_xyxy(box: Sequence[float]) -> List[float]:
    x, y, w, h = [float(v) for v in box]
    return [x, y, x + w, y + h]


def xyxy_to_coco_xywh(box: Sequence[float]) -> List[float]:
    x1, y1, x2, y2 = [float(v) for v in box]
    return [x1, y1, max(0.0, x2 - x1), max(0.0, y2 - y1)]


def clip_box_xyxy(box: Sequence[float], width: int, height: int) -> List[float]:
    x1, y1, x2, y2 = [float(v) for v in box]
    x1 = min(max(0.0, x1), float(width))
    y1 = min(max(0.0, y1), float(height))
    x2 = min(max(0.0, x2), float(width))
    y2 = min(max(0.0, y2), float(height))
    if x2 < x1:
        x1, x2 = x2, x1
    if y2 < y1:
        y1, y2 = y2, y1
    return [x1, y1, x2, y2]


def shift_box_xyxy(box: Sequence[float], dx: float, dy: float) -> List[float]:
    x1, y1, x2, y2 = [float(v) for v in box]
    return [x1 + dx, y1 + dy, x2 + dx, y2 + dy]


def box_iou_xyxy(a: Sequence[float], b: Sequence[float]) -> float:
    ax1, ay1, ax2, ay2 = [float(v) for v in a]
    bx1, by1, bx2, by2 = [float(v) for v in b]
    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)
    inter_w = max(0.0, inter_x2 - inter_x1)
    inter_h = max(0.0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    denom = area_a + area_b - inter_area
    if denom <= 0:
        return 0.0
    return inter_area / denom

## Dataset Loading

In [45]:
from datasets import load_dataset


def load_samples(config: EvalConfig) -> List[Sample]:
    ds = load_dataset(config.dataset_name, split=config.split)
    if config.max_samples is not None:
        ds = ds.select(range(min(config.max_samples, len(ds))))

    samples: List[Sample] = []
    for idx, row in enumerate(tqdm(ds, desc="Loading HF dataset")):
        image = row["image"].convert("RGB")
        objects = row["objects"]
        boxes_xyxy = [coco_xywh_to_xyxy(b) for b in objects["bbox"]]
        labels = [int(c) for c in objects["category"]]
        file_name = row.get("file_name", f"image_{idx:06d}.png")
        samples.append(
            Sample(
                image_id=idx,
                file_name=file_name,
                image=image,
                gt_boxes_xyxy=boxes_xyxy,
                gt_labels=labels,
            )
        )
    return samples

## Surya Prediction Mapping

This section:
- initializes Surya predictors
- applies table/layout heuristics
- converts structural outputs into unified `Prediction` objects.

In [46]:
def ensure_transformers_surya_compat() -> None:
    try:
        import torch
        from transformers import pytorch_utils as pu
    except Exception:
        return

    try:
        from transformers import modeling_utils as mu
    except Exception:
        mu = None

    if not hasattr(pu, "find_pruneable_heads_and_indices") and mu is not None:
        if hasattr(mu, "find_pruneable_heads_and_indices"):
            pu.find_pruneable_heads_and_indices = mu.find_pruneable_heads_and_indices

    if not hasattr(pu, "prune_linear_layer") and mu is not None:
        if hasattr(mu, "prune_linear_layer"):
            pu.prune_linear_layer = mu.prune_linear_layer

    if not hasattr(pu, "meshgrid"):
        pu.meshgrid = lambda *tensors, indexing="ij": torch.meshgrid(*tensors, indexing=indexing)


def init_surya_predictors(config: EvalConfig):
    ensure_transformers_surya_compat()
    from surya.settings import settings
    from surya.table_rec import TableRecPredictor

    table_predictor = TableRecPredictor(device=config.device or settings.TORCH_DEVICE_MODEL)
    layout_predictor = None

    if config.use_layout_table_detection:
        from surya.foundation import FoundationPredictor
        from surya.layout import LayoutPredictor

        foundation = FoundationPredictor(
            checkpoint=settings.LAYOUT_MODEL_CHECKPOINT,
            device=config.device or settings.TORCH_DEVICE_MODEL,
        )
        layout_predictor = LayoutPredictor(foundation)

    return table_predictor, layout_predictor


def _layout_boxes_to_table_bboxes(layout_result: Any, include_toc: bool) -> List[List[float]]:
    keep_labels = {"Table"}
    if include_toc:
        keep_labels.add("TableOfContents")
    return [box.bbox for box in layout_result.bboxes if box.label in keep_labels]


def _box_area_xyxy(box: Sequence[float]) -> float:
    x1, y1, x2, y2 = [float(v) for v in box]
    return max(0.0, x2 - x1) * max(0.0, y2 - y1)


def _intersection_area_xyxy(a: Sequence[float], b: Sequence[float]) -> float:
    ax1, ay1, ax2, ay2 = [float(v) for v in a]
    bx1, by1, bx2, by2 = [float(v) for v in b]
    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)
    return max(0.0, inter_x2 - inter_x1) * max(0.0, inter_y2 - inter_y1)


def _overlap_ratio_on_first_xyxy(a: Sequence[float], b: Sequence[float]) -> float:
    area_a = _box_area_xyxy(a)
    if area_a <= 0.0:
        return 0.0
    return _intersection_area_xyxy(a, b) / area_a


def _interval_union_length(intervals: List[Tuple[float, float]]) -> float:
    if not intervals:
        return 0.0
    intervals_sorted = sorted((float(a), float(b)) for a, b in intervals)
    total = 0.0
    cur_start, cur_end = intervals_sorted[0]
    for start, end in intervals_sorted[1:]:
        if start <= cur_end:
            cur_end = max(cur_end, end)
        else:
            total += max(0.0, cur_end - cur_start)
            cur_start, cur_end = start, end
    total += max(0.0, cur_end - cur_start)
    return total


def _split_contiguous_indices(indices: List[int]) -> List[List[int]]:
    if not indices:
        return []
    sorted_unique = sorted(set(int(i) for i in indices))
    runs: List[List[int]] = [[sorted_unique[0]]]
    for idx in sorted_unique[1:]:
        if idx == runs[-1][-1] + 1:
            runs[-1].append(idx)
        else:
            runs.append([idx])
    return runs


def _union_boxes_xyxy(boxes: List[List[float]]) -> List[float]:
    x1 = min(b[0] for b in boxes)
    y1 = min(b[1] for b in boxes)
    x2 = max(b[2] for b in boxes)
    y2 = max(b[3] for b in boxes)
    return [float(x1), float(y1), float(x2), float(y2)]


def _x_overlap_ratio_min_width(a: Sequence[float], b: Sequence[float]) -> float:
    ax1, _, ax2, _ = [float(v) for v in a]
    bx1, _, bx2, _ = [float(v) for v in b]
    inter = max(0.0, min(ax2, bx2) - max(ax1, bx1))
    wa = max(1e-6, ax2 - ax1)
    wb = max(1e-6, bx2 - bx1)
    return inter / min(wa, wb)


def _expand_box_xyxy(box: Sequence[float], image_width: int, image_height: int, expand_ratio: float) -> List[float]:
    if expand_ratio <= 0:
        return clip_box_xyxy(box, image_width, image_height)
    x1, y1, x2, y2 = [float(v) for v in box]
    w = max(0.0, x2 - x1)
    h = max(0.0, y2 - y1)
    dx = w * float(expand_ratio)
    dy = h * float(expand_ratio)
    return clip_box_xyxy([x1 - dx, y1 - dy, x2 + dx, y2 + dy], image_width, image_height)


def _find_best_structure_index(box_xyxy: Sequence[float], structure_boxes_xyxy: List[List[float]],
                               min_overlap_ratio: float = 0.05) -> Optional[int]:
    best_idx: Optional[int] = None
    best_overlap = 0.0
    for idx, sbox in enumerate(structure_boxes_xyxy):
        overlap = _overlap_ratio_on_first_xyxy(box_xyxy, sbox)
        if overlap > best_overlap:
            best_overlap = overlap
            best_idx = idx
    if best_idx is None or best_overlap < min_overlap_ratio:
        return None
    return best_idx


def _count_overlaps_on_first(box_xyxy: Sequence[float], candidate_boxes_xyxy: List[List[float]],
                             overlap_ratio_threshold: float) -> int:
    return sum(
        1 for cbox in candidate_boxes_xyxy if _overlap_ratio_on_first_xyxy(box_xyxy, cbox) >= overlap_ratio_threshold)


def _collect_overlap_indices(box_xyxy: Sequence[float], candidate_boxes_xyxy: List[List[float]],
                             overlap_ratio_threshold: float) -> List[int]:
    hits: List[int] = []
    for idx, cbox in enumerate(candidate_boxes_xyxy):
        if _overlap_ratio_on_first_xyxy(box_xyxy, cbox) >= overlap_ratio_threshold:
            hits.append(idx)
    return hits


def _grid_bbox_from_hit_indices(row_hit_indices: List[int], col_hit_indices: List[int], row_boxes: List[List[float]],
                                col_boxes: List[List[float]]) -> Optional[List[float]]:
    if not row_hit_indices or not col_hit_indices:
        return None
    r1 = min(int(i) for i in row_hit_indices)
    r2 = max(int(i) for i in row_hit_indices)
    c1 = min(int(i) for i in col_hit_indices)
    c2 = max(int(i) for i in col_hit_indices)
    return [
        float(col_boxes[c1][0]),
        float(row_boxes[r1][1]),
        float(col_boxes[c2][2]),
        float(row_boxes[r2][3]),
    ]


def _deduplicate_predictions(
        predictions: List[Prediction],
        iou_threshold: float,
        span_iou_threshold: Optional[float] = None,
) -> List[Prediction]:
    deduped: List[Prediction] = []
    span_class_id = CLASS_NAME_TO_ID["table spanning cell"]
    for class_id in sorted({p.category_id for p in predictions}):
        class_preds = [p for p in predictions if p.category_id == class_id]
        class_preds.sort(key=lambda p: float(p.score), reverse=True)
        kept: List[Prediction] = []
        class_iou_threshold = iou_threshold
        if span_iou_threshold is not None and class_id == span_class_id:
            class_iou_threshold = float(span_iou_threshold)
        for pred in class_preds:
            if not any(box_iou_xyxy(pred.bbox_xyxy, k.bbox_xyxy) >= class_iou_threshold for k in kept):
                kept.append(pred)
        deduped.extend(kept)
    return deduped


def _build_relaxed_vertical_span_boxes(unmerged_items: List[Dict[str, Any]], min_x_overlap_ratio: float,
                                       max_row_gap: int) -> List[Dict[str, Any]]:
    by_row: Dict[int, List[int]] = {}
    for idx, item in enumerate(unmerged_items):
        row_id = item.get("row_id")
        if row_id is not None:
            by_row.setdefault(int(row_id), []).append(idx)

    recovered: List[Dict[str, Any]] = []
    consumed: set[int] = set()

    for idx, item in enumerate(unmerged_items):
        if idx in consumed or not bool(item.get("merge_down", False)) or item.get("row_id") is None:
            continue

        chain = [idx]
        cur_idx = idx
        cur_row = int(item["row_id"])

        while True:
            cur_item = unmerged_items[cur_idx]
            if not bool(cur_item.get("merge_down", False)):
                break

            candidates: List[Tuple[float, int, int]] = []
            for row_gap in range(1, max(1, int(max_row_gap)) + 1):
                for nxt_idx in by_row.get(cur_row + row_gap, []):
                    nxt_item = unmerged_items[nxt_idx]
                    if not bool(nxt_item.get("merge_up", False)):
                        continue
                    x_overlap = _x_overlap_ratio_min_width(cur_item["bbox"], nxt_item["bbox"])
                    if x_overlap >= min_x_overlap_ratio:
                        candidates.append((-x_overlap, row_gap, nxt_idx))

            if not candidates:
                break

            candidates.sort()
            nxt_idx = candidates[0][2]
            chain.append(nxt_idx)
            cur_idx = nxt_idx
            nxt_row = unmerged_items[cur_idx].get("row_id")
            if nxt_row is None:
                break
            cur_row = int(nxt_row)

        if len(chain) >= 2:
            for used_idx in chain:
                consumed.add(used_idx)
            chain_boxes = [unmerged_items[c]["bbox"] for c in chain]
            conf = float(np.mean([float(unmerged_items[c]["confidence"]) for c in chain]))
            recovered.append({"bbox": _union_boxes_xyxy(chain_boxes), "confidence": conf,
                              "source": "heuristic/spanning_cell_vertical_chain"})

    return recovered


def _map_table_result_to_predictions(
        image_id: int,
        image_size: Tuple[int, int],
        table_bbox_xyxy: Sequence[float],
        crop_bbox_xyxy: Sequence[float],
        table_result: Any,
        config: EvalConfig,
) -> List[Prediction]:
    width, height = image_size
    table_bbox_xyxy = clip_box_xyxy(table_bbox_xyxy, width, height)
    tx1, ty1, _, _ = [float(v) for v in crop_bbox_xyxy]

    preds: List[Prediction] = [
        Prediction(image_id=image_id, bbox_xyxy=table_bbox_xyxy, category_id=CLASS_NAME_TO_ID["table"], score=1.0,
                   source="layout/table_bbox")
    ]

    rows = list(getattr(table_result, "rows", []) or [])
    cols = list(getattr(table_result, "cols", []) or [])
    cells_merged = list(getattr(table_result, "cells", []) or [])
    cells_unmerged = list(getattr(table_result, "unmerged_cells", []) or cells_merged)

    row_boxes_global: List[List[float]] = []
    row_heights: List[float] = []
    for row in rows:
        gbox = clip_box_xyxy(shift_box_xyxy(row.bbox, tx1, ty1), width, height)
        row_boxes_global.append(gbox)
        row_heights.append(max(1e-6, gbox[3] - gbox[1]))
        preds.append(Prediction(image_id=image_id, bbox_xyxy=gbox, category_id=CLASS_NAME_TO_ID["table row"],
                                score=float(row.confidence) if row.confidence is not None else 1.0, source="surya/row"))

    col_boxes_global: List[List[float]] = []
    col_widths: List[float] = []
    for col in cols:
        gbox = clip_box_xyxy(shift_box_xyxy(col.bbox, tx1, ty1), width, height)
        col_boxes_global.append(gbox)
        col_widths.append(max(1e-6, gbox[2] - gbox[0]))
        preds.append(Prediction(image_id=image_id, bbox_xyxy=gbox, category_id=CLASS_NAME_TO_ID["table column"],
                                score=float(col.confidence) if col.confidence is not None else 1.0, source="surya/col"))

    def _to_cell_item(cell: Any) -> Dict[str, Any]:
        bbox = clip_box_xyxy(shift_box_xyxy(cell.bbox, tx1, ty1), width, height)
        return {
            "bbox": bbox,
            "is_header": bool(getattr(cell, "is_header", False)),
            "confidence": float(cell.confidence) if getattr(cell, "confidence", None) is not None else 1.0,
            "rowspan": int(cell.rowspan) if getattr(cell, "rowspan", None) is not None else 1,
            "colspan": int(cell.colspan) if getattr(cell, "colspan", None) is not None else 1,
            "row_id": int(cell.row_id) if getattr(cell, "row_id", None) is not None else None,
            "col_id": int(cell.col_id) if getattr(cell, "col_id", None) is not None else None,
            "merge_up": bool(getattr(cell, "merge_up", False)),
            "merge_down": bool(getattr(cell, "merge_down", False)),
        }

    merged_items = [_to_cell_item(c) for c in cells_merged]
    unmerged_items = [_to_cell_item(c) for c in cells_unmerged]

    row_cell_counts = [0 for _ in rows]
    row_header_counts = [0 for _ in rows]
    row_header_conf_sum = [0.0 for _ in rows]
    row_header_intervals: List[List[Tuple[float, float]]] = [[] for _ in rows]
    row_max_cell_width = [0.0 for _ in rows]

    for item in merged_items:
        if not row_boxes_global:
            continue
        row_idx = _find_best_structure_index(item["bbox"], row_boxes_global, min_overlap_ratio=0.05)
        if row_idx is None:
            continue
        row_cell_counts[row_idx] += 1
        row_max_cell_width[row_idx] = max(row_max_cell_width[row_idx], max(0.0, item["bbox"][2] - item["bbox"][0]))
        if item["is_header"]:
            row_header_counts[row_idx] += 1
            row_header_conf_sum[row_idx] += item["confidence"]
            row_header_intervals[row_idx].append((item["bbox"][0], item["bbox"][2]))

    row_header_width_fraction = [0.0 for _ in rows]
    for idx in range(len(rows)):
        row_width = max(1e-6, row_boxes_global[idx][2] - row_boxes_global[idx][0])
        row_header_width_fraction[idx] = min(1.0, _interval_union_length(row_header_intervals[idx]) / row_width)

    header_candidate_rows: set[int] = set()
    if config.use_cell_based_header_mapping:
        for idx in range(len(rows)):
            row_header_flag = bool(getattr(rows[idx], "is_header", False))
            header_ratio = row_header_counts[idx] / max(1, row_cell_counts[idx])
            width_frac = row_header_width_fraction[idx]
            qualifies = row_header_counts[
                            idx] > 0 and header_ratio >= config.header_row_cell_fraction_threshold and width_frac >= config.header_row_min_width_fraction
            if row_header_flag and width_frac >= max(0.35, config.header_row_min_width_fraction * 0.6):
                qualifies = True
            if qualifies:
                header_candidate_rows.add(idx)

    top_header_rows: set[int] = set()
    if header_candidate_rows:
        started = False
        for idx in range(len(rows)):
            if idx in header_candidate_rows:
                top_header_rows.add(idx)
                started = True
                if len(top_header_rows) >= max(1, int(config.max_top_header_rows)):
                    break
            elif started:
                break
    if not top_header_rows and rows and (bool(getattr(rows[0], "is_header", False)) or row_header_counts[0] > 0):
        top_header_rows.add(0)

    header_row_scores: Dict[int, float] = {}
    for idx in sorted(top_header_rows):
        row_conf = float(rows[idx].confidence) if rows[idx].confidence is not None else 1.0
        if row_header_counts[idx] > 0:
            row_conf = max(row_conf, row_header_conf_sum[idx] / row_header_counts[idx])
        header_row_scores[idx] = row_conf

    if config.emit_column_header_band_union and top_header_rows:
        for run in _split_contiguous_indices(list(top_header_rows)):
            preds.append(Prediction(image_id=image_id, bbox_xyxy=_union_boxes_xyxy([row_boxes_global[i] for i in run]),
                                    category_id=CLASS_NAME_TO_ID["table column header"],
                                    score=max(header_row_scores[i] for i in run),
                                    source="heuristic/header_top_band_union"))
    else:
        for idx in sorted(top_header_rows):
            preds.append(Prediction(image_id=image_id, bbox_xyxy=row_boxes_global[idx],
                                    category_id=CLASS_NAME_TO_ID["table column header"], score=header_row_scores[idx],
                                    source="heuristic/header_top_band_row"))

    projected_row_header_rows: set[int] = set()
    if config.map_projected_row_header_from_non_top_header_rows:
        projected_row_header_rows.update(idx for idx in header_candidate_rows if idx not in top_header_rows)

    table_width = max(1.0, float(table_bbox_xyxy[2] - table_bbox_xyxy[0]))
    if config.enable_projected_row_header_geom_fallback:
        for idx in range(len(rows)):
            if idx in top_header_rows or idx in projected_row_header_rows:
                continue
            if row_cell_counts[idx] <= 0 or row_cell_counts[idx] > max(1, int(config.projected_row_header_max_cells)):
                continue
            if (row_max_cell_width[idx] / table_width) >= config.projected_row_header_min_width_fraction:
                projected_row_header_rows.add(idx)

    for idx in sorted(projected_row_header_rows):
        row_conf = float(rows[idx].confidence) if rows[idx].confidence is not None else 1.0
        if row_header_counts[idx] > 0:
            row_conf = max(row_conf, row_header_conf_sum[idx] / row_header_counts[idx])
        preds.append(Prediction(image_id=image_id, bbox_xyxy=row_boxes_global[idx],
                                category_id=CLASS_NAME_TO_ID["table projected row header"], score=row_conf,
                                source="heuristic/projected_row_header"))

    span_preds: List[Prediction] = []
    table_area = max(1e-6, _box_area_xyxy(table_bbox_xyxy))
    span_class_id = CLASS_NAME_TO_ID["table spanning cell"]

    span_debug_counts: Dict[str, int] = {
        "raw_structural": 0,
        "rejected_area": 0,
        "rejected_grid": 0,
        "kept_grid_rebuilt": 0,
        "kept_raw_fallback": 0,
        "dedup_dropped": 0,
    }

    def _span_candidate(
            raw_box: Sequence[float],
            confidence: float,
            source: str,
            require_structural_limits: bool,
    ) -> Optional[Prediction]:
        clipped_raw = clip_box_xyxy(raw_box, width, height)
        area_ratio = _box_area_xyxy(clipped_raw) / table_area
        if area_ratio > float(config.span_max_area_fraction_of_table):
            span_debug_counts["rejected_area"] += 1
            return None

        row_hit_indices = _collect_overlap_indices(
            clipped_raw,
            row_boxes_global,
            config.span_relaxed_row_overlap_ratio_threshold,
        ) if row_boxes_global else []
        col_hit_indices = _collect_overlap_indices(
            clipped_raw,
            col_boxes_global,
            config.span_relaxed_col_overlap_ratio_threshold,
        ) if col_boxes_global else []

        row_hits = len(row_hit_indices)
        col_hits = len(col_hit_indices)
        if row_hits <= 1 and col_hits <= 1:
            span_debug_counts["rejected_grid"] += 1
            return None

        final_box = clipped_raw
        if config.span_use_grid_rebuilt_boxes:
            grid_box = _grid_bbox_from_hit_indices(row_hit_indices, col_hit_indices, row_boxes_global, col_boxes_global)
            if grid_box is not None:
                final_box = clip_box_xyxy(grid_box, width, height)
                span_debug_counts["kept_grid_rebuilt"] += 1
            else:
                span_debug_counts["kept_raw_fallback"] += 1

        score = float(confidence) + float(config.span_grid_score_alpha) * float(row_hits + col_hits)
        return Prediction(
            image_id=image_id,
            bbox_xyxy=final_box,
            category_id=span_class_id,
            score=score,
            source=source,
        )

    if config.use_merged_structural_spans:
        for item in merged_items:
            if item["rowspan"] > 1 or item["colspan"] > 1:
                if item["rowspan"] <= int(config.structural_span_max_rowspan) and item["colspan"] <= int(
                        config.structural_span_max_colspan):
                    span_debug_counts["raw_structural"] += 1
                    candidate = _span_candidate(
                        raw_box=item["bbox"],
                        confidence=float(item["confidence"]),
                        source="surya/spanning_cell_structural_raw",
                        require_structural_limits=True,
                    )
                    if candidate is not None:
                        span_preds.append(candidate)

    if config.use_unmerged_structural_spans:
        for item in unmerged_items:
            if item["rowspan"] > 1 or item["colspan"] > 1:
                if item["rowspan"] <= int(config.structural_span_max_rowspan) and item["colspan"] <= int(
                        config.structural_span_max_colspan):
                    span_debug_counts["raw_structural"] += 1
                    candidate = _span_candidate(
                        raw_box=item["bbox"],
                        confidence=float(item["confidence"]),
                        source="surya/spanning_cell_unmerged_raw",
                        require_structural_limits=True,
                    )
                    if candidate is not None:
                        span_preds.append(candidate)

    if config.enable_unmerged_vertical_span_recovery and unmerged_items:
        recovered_vertical = _build_relaxed_vertical_span_boxes(
            unmerged_items,
            min_x_overlap_ratio=config.vertical_span_min_x_overlap_ratio,
            max_row_gap=config.vertical_span_max_row_gap,
        )
        for rec in recovered_vertical:
            candidate = _span_candidate(
                raw_box=rec["bbox"],
                confidence=float(rec["confidence"]),
                source=rec["source"],
                require_structural_limits=False,
            )
            if candidate is not None:
                span_preds.append(candidate)

    if config.enable_span_geometric_fallback and row_boxes_global and col_boxes_global:
        for item in unmerged_items:
            if item["rowspan"] > 1 or item["colspan"] > 1:
                continue
            row_hits = _count_overlaps_on_first(item["bbox"], row_boxes_global,
                                                config.span_relaxed_row_overlap_ratio_threshold)
            col_hits = _count_overlaps_on_first(item["bbox"], col_boxes_global,
                                                config.span_relaxed_col_overlap_ratio_threshold)
            row_idx = _find_best_structure_index(item["bbox"], row_boxes_global, min_overlap_ratio=0.01)
            col_idx = _find_best_structure_index(item["bbox"], col_boxes_global, min_overlap_ratio=0.01)
            cell_h = max(1e-6, item["bbox"][3] - item["bbox"][1])
            cell_w = max(1e-6, item["bbox"][2] - item["bbox"][0])
            row_h = row_heights[row_idx] if row_idx is not None else cell_h
            col_w = col_widths[col_idx] if col_idx is not None else cell_w

            relaxed_rowspan = row_hits > 1 and (cell_h / max(1e-6, row_h)) >= config.span_min_row_height_ratio
            relaxed_colspan = col_hits > 1 and (cell_w / max(1e-6, col_w)) >= config.span_min_col_width_ratio

            if relaxed_rowspan or relaxed_colspan:
                candidate = _span_candidate(
                    raw_box=item["bbox"],
                    confidence=float(item["confidence"]) * 0.9,
                    source="heuristic/spanning_cell_relaxed_geom",
                    require_structural_limits=False,
                )
                if candidate is not None:
                    span_preds.append(candidate)

    preds.extend(span_preds)
    return preds

## Batched Inference

Runs layout + table recognition in chunks to control memory usage and then maps model outputs back to image coordinates.

In [47]:
def run_surya_inference(
        samples: List[Sample],
        config: EvalConfig,
        table_predictor: Any,
        layout_predictor: Any = None,
) -> Dict[int, List[Prediction]]:
    import gc

    def _clear_cuda_cache_if_available() -> None:
        try:
            import torch

            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception:
            pass

    predictions_by_image: Dict[int, List[Prediction]] = {s.image_id: [] for s in samples}

    image_size_by_id: Dict[int, Tuple[int, int]] = {}
    for s in samples:
        if s.image is None:
            raise ValueError(
                "Sample.image is None before inference. Disable release_images_after_inference or reload samples."
            )
        image_size_by_id[s.image_id] = s.image.size

    layout_bs = int(config.layout_batch_size) if config.layout_batch_size else 2
    table_bs = int(config.table_rec_batch_size) if config.table_rec_batch_size else 8
    sample_chunk_size = max(1, int(config.inference_chunk_size))
    max_crops_per_pass = max(1, int(config.max_table_crops_per_pass))

    chunk_ranges = range(0, len(samples), sample_chunk_size)
    for chunk_start in tqdm(chunk_ranges, desc="Inference chunks"):
        chunk_samples = samples[chunk_start: chunk_start + sample_chunk_size]
        chunk_images = [s.image for s in chunk_samples]

        # 1) Detect table regions for this chunk
        table_regions_by_image: Dict[int, List[List[float]]] = {s.image_id: [] for s in chunk_samples}
        if layout_predictor is not None:
            layout_results = layout_predictor(chunk_images, batch_size=layout_bs)
            for sample, layout_result in zip(chunk_samples, layout_results):
                table_boxes = _layout_boxes_to_table_bboxes(
                    layout_result,
                    include_toc=config.include_table_of_contents_as_table,
                )
                if not table_boxes and config.fallback_to_full_image_when_no_table:
                    w, h = image_size_by_id[sample.image_id]
                    table_boxes = [[0.0, 0.0, float(w), float(h)]]
                table_regions_by_image[sample.image_id] = table_boxes
        else:
            for sample in chunk_samples:
                w, h = image_size_by_id[sample.image_id]
                table_regions_by_image[sample.image_id] = [[0.0, 0.0, float(w), float(h)]]

        # 2) Crop table regions for this chunk
        all_crops: List[Image.Image] = []
        crop_meta: List[Tuple[int, List[float], List[float]]] = []

        for sample in chunk_samples:
            sample_image = sample.image
            if sample_image is None:
                raise ValueError(f"Missing image for sample {sample.image_id}")

            image_w, image_h = image_size_by_id[sample.image_id]
            for box in table_regions_by_image[sample.image_id]:
                table_box = clip_box_xyxy(box, image_w, image_h)
                crop_box = _expand_box_xyxy(
                    table_box,
                    image_width=image_w,
                    image_height=image_h,
                    expand_ratio=config.table_crop_expand_ratio,
                )

                x1, y1, x2, y2 = crop_box
                left = int(math.floor(x1))
                top = int(math.floor(y1))
                right = int(math.ceil(x2))
                bottom = int(math.ceil(y2))

                if right <= left or bottom <= top:
                    continue

                crop = sample_image.crop((left, top, right, bottom))
                all_crops.append(crop)
                crop_meta.append((sample.image_id, table_box, [float(left), float(top), float(right), float(bottom)]))

            if config.release_images_after_inference and not config.save_visualizations:
                sample.image = None

        # 3) Run table recognition in bounded passes
        for crop_start in range(0, len(all_crops), max_crops_per_pass):
            crop_end = crop_start + max_crops_per_pass
            crops_batch = all_crops[crop_start:crop_end]
            meta_batch = crop_meta[crop_start:crop_end]

            if not crops_batch:
                continue

            table_results = table_predictor(crops_batch, batch_size=table_bs)
            for (image_id, table_bbox, crop_bbox), table_result in zip(meta_batch, table_results):
                mapped = _map_table_result_to_predictions(
                    image_id=image_id,
                    image_size=image_size_by_id[image_id],
                    table_bbox_xyxy=table_bbox,
                    crop_bbox_xyxy=crop_bbox,
                    table_result=table_result,
                    config=config,
                )
                predictions_by_image[image_id].extend(mapped)

            for crop in crops_batch:
                try:
                    crop.close()
                except Exception:
                    pass

            del table_results

        del chunk_images
        del table_regions_by_image
        del all_crops
        del crop_meta
        gc.collect()
        if config.clear_cuda_cache_between_chunks:
            _clear_cuda_cache_if_available()

    if config.dedupe_predictions:
        for image_id in predictions_by_image:
            predictions_by_image[image_id] = _deduplicate_predictions(
                predictions_by_image[image_id],
                iou_threshold=config.dedupe_iou_threshold,
                span_iou_threshold=config.span_dedup_iou,
            )

    return predictions_by_image

## Evaluation (TorchMetrics)

This section computes:
- COCO-style detection metrics with `MeanAveragePrecision`
- detection-level `precision`, `recall`, and `f1` at IoU=0.50 from TP/FP/FN counts
- per-class `precision`, `recall`, and `f1` at IoU=0.50 for easier analysis.

In [48]:
def evaluate_predictions(
        samples: List[Sample],
        predictions_by_image: Dict[int, List[Prediction]],
        config: EvalConfig,
) -> Dict[str, Any]:
    def to_float(value: Any) -> float:
        if value is None:
            return math.nan
        if isinstance(value, torch.Tensor):
            if value.numel() == 0:
                return math.nan
            return float(value.detach().cpu().item())
        try:
            return float(value)
        except Exception:
            return math.nan

    def to_float_tensor(values: List[float], shape: Tuple[int, ...]) -> torch.Tensor:
        if not values:
            return torch.zeros(shape, dtype=torch.float32)
        return torch.tensor(values, dtype=torch.float32)

    def to_int_tensor(values: List[int]) -> torch.Tensor:
        if not values:
            return torch.zeros((0,), dtype=torch.int64)
        return torch.tensor(values, dtype=torch.int64)

    def safe_div(numerator: float, denominator: float) -> float:
        return float(numerator / denominator) if denominator > 0 else 0.0

    def compute_detection_prf1(iou_threshold: float = 0.50) -> Dict[str, Any]:
        total_tp = 0
        total_fp = 0
        total_fn = 0

        per_class_counts: Dict[int, Dict[str, int]] = {
            class_id: {"tp": 0, "fp": 0, "fn": 0}
            for class_id in range(len(CLASS_NAMES))
        }

        for sample in samples:
            gt_by_class: Dict[int, List[List[float]]] = {c: [] for c in range(len(CLASS_NAMES))}
            pred_by_class: Dict[int, List[Tuple[float, List[float]]]] = {c: [] for c in range(len(CLASS_NAMES))}

            for gt_box, gt_label in zip(sample.gt_boxes_xyxy, sample.gt_labels):
                class_id = int(gt_label)
                if 0 <= class_id < len(CLASS_NAMES):
                    gt_by_class[class_id].append(gt_box)

            for pred in predictions_by_image.get(sample.image_id, []):
                class_id = int(pred.category_id)
                if 0 <= class_id < len(CLASS_NAMES):
                    pred_by_class[class_id].append((float(pred.score), pred.bbox_xyxy))

            for class_id in range(len(CLASS_NAMES)):
                gt_boxes = gt_by_class[class_id]
                class_preds = sorted(pred_by_class[class_id], key=lambda x: x[0], reverse=True)
                matched_gt: set[int] = set()

                tp = 0
                for _, pred_box in class_preds:
                    best_gt_idx = -1
                    best_iou = 0.0
                    for gt_idx, gt_box in enumerate(gt_boxes):
                        if gt_idx in matched_gt:
                            continue
                        iou = box_iou_xyxy(pred_box, gt_box)
                        if iou > best_iou:
                            best_iou = iou
                            best_gt_idx = gt_idx

                    if best_gt_idx >= 0 and best_iou >= iou_threshold:
                        matched_gt.add(best_gt_idx)
                        tp += 1

                fp = len(class_preds) - tp
                fn = len(gt_boxes) - tp

                per_class_counts[class_id]["tp"] += tp
                per_class_counts[class_id]["fp"] += fp
                per_class_counts[class_id]["fn"] += fn

                total_tp += tp
                total_fp += fp
                total_fn += fn

        precision = safe_div(total_tp, total_tp + total_fp)
        recall = safe_div(total_tp, total_tp + total_fn)
        f1 = safe_div(2 * precision * recall, precision + recall)

        per_class_prf1: Dict[int, Dict[str, float]] = {}
        for class_id in range(len(CLASS_NAMES)):
            tp = per_class_counts[class_id]["tp"]
            fp = per_class_counts[class_id]["fp"]
            fn = per_class_counts[class_id]["fn"]
            p = safe_div(tp, tp + fp)
            r = safe_div(tp, tp + fn)
            per_class_prf1[class_id] = {
                "tp": tp,
                "fp": fp,
                "fn": fn,
                "precision": p,
                "recall": r,
                "f1": safe_div(2 * p * r, p + r),
            }

        return {
            "overall": {
                "precision": precision,
                "recall": recall,
                "f1": f1,
                "tp": total_tp,
                "fp": total_fp,
                "fn": total_fn,
            },
            "per_class": per_class_prf1,
        }

    preds: List[Dict[str, torch.Tensor]] = []
    targets: List[Dict[str, torch.Tensor]] = []

    for sample in samples:
        targets.append(
            {
                "boxes": to_float_tensor(sample.gt_boxes_xyxy, (0, 4)).reshape(-1, 4),
                "labels": to_int_tensor(sample.gt_labels),
            }
        )

        sample_preds = predictions_by_image.get(sample.image_id, [])
        preds.append(
            {
                "boxes": to_float_tensor([p.bbox_xyxy for p in sample_preds], (0, 4)).reshape(-1, 4),
                "scores": to_float_tensor([float(p.score) for p in sample_preds], (0,)),
                "labels": to_int_tensor([int(p.category_id) for p in sample_preds]),
            }
        )

    def compute_map(iou_thresholds: Sequence[float]) -> Dict[str, Any]:
        metric = MeanAveragePrecision(
            box_format="xyxy",
            iou_type="bbox",
            class_metrics=True,
            iou_thresholds=[float(t) for t in iou_thresholds],
        )
        metric.update(preds, targets)
        return metric.compute()

    def get_per_class(output: Dict[str, Any], key: str) -> Dict[int, float]:
        classes = output.get("classes")
        values = output.get(key)
        if classes is None or values is None:
            return {}

        classes_list = classes.detach().cpu().tolist() if isinstance(classes, torch.Tensor) else list(classes)
        values_list = values.detach().cpu().tolist() if isinstance(values, torch.Tensor) else list(values)
        return {int(class_id): float(value) for class_id, value in zip(classes_list, values_list)}

    full_metrics = compute_map(config.map_thresholds)

    ap_by_threshold: Dict[float, float] = {}
    per_class_ap_by_threshold: Dict[str, Dict[str, float]] = {class_name: {} for class_name in CLASS_NAMES}

    for threshold in config.map_thresholds:
        threshold_value = float(threshold)
        threshold_metrics = compute_map([threshold_value])

        ap_by_threshold[threshold_value] = to_float(threshold_metrics.get("map"))
        threshold_per_class = get_per_class(threshold_metrics, "map_per_class")

        for class_id, class_name in enumerate(CLASS_NAMES):
            per_class_ap_by_threshold[class_name][f"{threshold_value:.2f}"] = threshold_per_class.get(
                class_id,
                math.nan,
            )

    per_class_map_50_95_by_id = get_per_class(full_metrics, "map_per_class")
    per_class_mar_100_by_id = get_per_class(full_metrics, "mar_100_per_class")
    prf1_stats = compute_detection_prf1(iou_threshold=0.50)

    class_rows: List[Dict[str, Any]] = []
    for class_id, class_name in enumerate(CLASS_NAMES):
        class_prf1 = prf1_stats["per_class"].get(class_id, {})
        class_rows.append(
            {
                "class_id": class_id,
                "class_name": class_name,
                "AP@0.50": per_class_ap_by_threshold[class_name].get("0.50", math.nan),
                "mAP@0.50:0.95": per_class_map_50_95_by_id.get(class_id, math.nan),
                "mAR@100": per_class_mar_100_by_id.get(class_id, math.nan),
                "precision": class_prf1.get("precision", 0.0),
                "recall": class_prf1.get("recall", 0.0),
                "f1": class_prf1.get("f1", 0.0),
                "tp": class_prf1.get("tp", 0),
                "fp": class_prf1.get("fp", 0),
                "fn": class_prf1.get("fn", 0),
            }
        )

    class_df = pd.DataFrame(class_rows)

    map_50 = to_float(full_metrics.get("map_50"))
    if not np.isfinite(map_50):
        map_50 = ap_by_threshold.get(0.50, math.nan)

    summary = {
        "mAP@0.50": map_50,
        "mAP@0.50:0.95": to_float(full_metrics.get("map")),
        "mAP@0.75": to_float(full_metrics.get("map_75")),
        "mAR@1": to_float(full_metrics.get("mar_1")),
        "mAR@10": to_float(full_metrics.get("mar_10")),
        "mAR@100": to_float(full_metrics.get("mar_100")),
        "precision": prf1_stats["overall"]["precision"],
        "recall": prf1_stats["overall"]["recall"],
        "f1": prf1_stats["overall"]["f1"],
    }

    per_class_map_50_95 = {
        CLASS_NAMES[class_id]: per_class_map_50_95_by_id.get(class_id, math.nan)
        for class_id in range(len(CLASS_NAMES))
    }

    per_class_prf1 = {
        CLASS_NAMES[class_id]: {
            "precision": prf1_stats["per_class"][class_id]["precision"],
            "recall": prf1_stats["per_class"][class_id]["recall"],
            "f1": prf1_stats["per_class"][class_id]["f1"],
            "tp": prf1_stats["per_class"][class_id]["tp"],
            "fp": prf1_stats["per_class"][class_id]["fp"],
            "fn": prf1_stats["per_class"][class_id]["fn"],
        }
        for class_id in range(len(CLASS_NAMES))
    }

    return {
        "summary": summary,
        "class_metrics": class_df,
        "ap_by_threshold": ap_by_threshold,
        "per_class_map_50_95": per_class_map_50_95,
        "per_class_ap_by_threshold": per_class_ap_by_threshold,
        "per_class_prf1": per_class_prf1,
    }

## Export and Visualization Helpers

These helpers serialize predictions and optionally draw GT vs prediction overlays for qualitative checks.

In [49]:
def save_predictions_json(
        samples: List[Sample],
        predictions_by_image: Dict[int, List[Prediction]],
        out_path: Path,
) -> None:
    payload = []
    for s in samples:
        preds = predictions_by_image.get(s.image_id, [])
        payload.append(
            {
                "image_id": s.image_id,
                "file_name": s.file_name,
                "predictions": [
                    {
                        "bbox_xyxy": p.bbox_xyxy,
                        "bbox_xywh": xyxy_to_coco_xywh(p.bbox_xyxy),
                        "category_id": p.category_id,
                        "category_name": CLASS_NAMES[p.category_id],
                        "score": p.score,
                        "source": p.source,
                    }
                    for p in preds
                ],
            }
        )
    with out_path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)


def save_visualizations(
        samples: List[Sample],
        predictions_by_image: Dict[int, List[Prediction]],
        out_dir: Path,
        max_images: int,
) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)

    for s in samples[:max_images]:
        gt_img = s.image.copy()
        gt_draw = ImageDraw.Draw(gt_img)
        for box, class_id in zip(s.gt_boxes_xyxy, s.gt_labels):
            x1, y1, x2, y2 = box
            gt_draw.rectangle((x1, y1, x2, y2), outline="green", width=2)
            gt_draw.text((x1 + 2, y1 + 2), f"GT:{CLASS_NAMES[class_id]}", fill="green")

        pred_img = s.image.copy()
        pred_draw = ImageDraw.Draw(pred_img)
        for pred in predictions_by_image.get(s.image_id, []):
            x1, y1, x2, y2 = pred.bbox_xyxy
            pred_draw.rectangle((x1, y1, x2, y2), outline="red", width=2)
            pred_draw.text(
                (x1 + 2, y1 + 2),
                f"P:{CLASS_NAMES[pred.category_id]}:{pred.score:.2f}",
                fill="red",
            )

        gt_img.save(out_dir / f"{s.image_id:06d}_gt.png")
        pred_img.save(out_dir / f"{s.image_id:06d}_pred.png")

## Pipeline Orchestration

`run_experiment` is the main entry point. It handles end-to-end flow and writes JSON/CSV artifacts for quick inspection.

In [50]:
def run_experiment(config: EvalConfig) -> Dict[str, Any]:
    np.random.seed(config.random_seed)

    output_dir = Path(config.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    with (output_dir / "config.json").open("w", encoding="utf-8") as f:
        json.dump(asdict(config), f, indent=2)

    samples = load_samples(config)

    table_predictor, layout_predictor = init_surya_predictors(config)

    predictions_by_image = run_surya_inference(
        samples=samples,
        config=config,
        table_predictor=table_predictor,
        layout_predictor=layout_predictor,
    )

    eval_out = evaluate_predictions(samples, predictions_by_image, config)
    summary = eval_out["summary"]
    class_df: pd.DataFrame = eval_out["class_metrics"]

    save_predictions_json(samples, predictions_by_image, output_dir / "predictions.json")
    class_df.to_csv(output_dir / "class_metrics.csv", index=False)

    with (output_dir / "summary.json").open("w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)
    with (output_dir / "map_thresholds.json").open("w", encoding="utf-8") as f:
        json.dump(eval_out["ap_by_threshold"], f, indent=2)
    with (output_dir / "per_class_map_50_95.json").open("w", encoding="utf-8") as f:
        json.dump(eval_out["per_class_map_50_95"], f, indent=2)
    with (output_dir / "per_class_ap_by_threshold.json").open("w", encoding="utf-8") as f:
        json.dump(eval_out["per_class_ap_by_threshold"], f, indent=2)
    with (output_dir / "per_class_prf1.json").open("w", encoding="utf-8") as f:
        json.dump(eval_out["per_class_prf1"], f, indent=2)

    if config.save_visualizations:
        save_visualizations(
            samples=samples,
            predictions_by_image=predictions_by_image,
            out_dir=output_dir / "visualizations",
            max_images=config.num_visualizations,
        )

    print("Summary metrics:")
    print(json.dumps(summary, indent=2))
    print("\nPer-class mAP@0.50:0.95:")
    print(json.dumps(eval_out["per_class_map_50_95"], indent=2))
    print("\nPer-class precision/recall/F1:")
    print(json.dumps(eval_out["per_class_prf1"], indent=2))
    print("\nClass metrics:")
    print(class_df)

    return {
        "config": config,
        "samples": samples,
        "predictions_by_image": predictions_by_image,
        "evaluation": eval_out,
    }

In [53]:
# Class 3: reduce false-positive header rows
CONFIG.max_top_header_rows = 2
CONFIG.header_row_cell_fraction_threshold = 0.45
CONFIG.header_row_min_width_fraction = 0.75
CONFIG.emit_column_header_band_union = True

## Run the Demo

This cell executes the full pipeline:
1. load samples
2. run Surya inference
3. compute TorchMetrics detection metrics
4. save outputs under `CONFIG.output_dir`.

In [54]:
CONFIG.max_samples = 20
results = run_experiment(CONFIG)

Loading HF dataset:   0%|          | 0/20 [00:00<?, ?it/s]

Inference chunks:   0%|          | 0/1 [00:00<?, ?it/s]


Recognizing Layout: 100%|██████████| 20/20 [00:14<00:00,  1.36it/s]

Recognizing tables: 100%|██████████| 3/3 [00:04<00:00,  1.36s/it]


Summary metrics:
{
  "mAP@0.50": 0.716094970703125,
  "mAP@0.50:0.95": 0.5324313640594482,
  "mAP@0.75": 0.5935181379318237,
  "mAR@1": 0.2998698651790619,
  "mAR@10": 0.5495339035987854,
  "mAR@100": 0.6054474115371704,
  "precision": 0.9574944071588367,
  "recall": 0.9224137931034483,
  "f1": 0.9396267837541165
}

Per-class mAP@0.50:0.95:
{
  "table": 0.6891818046569824,
  "table column": 0.8985154032707214,
  "table row": 0.6795135140419006,
  "table column header": 0.6018751859664917,
  "table projected row header": 0.0,
  "table spanning cell": 0.3255021274089813
}

Per-class precision/recall/F1:
{
  "table": {
    "precision": 0.9545454545454546,
    "recall": 1.0,
    "f1": 0.9767441860465117,
    "tp": 21,
    "fp": 1,
    "fn": 0
  },
  "table column": {
    "precision": 0.9714285714285714,
    "recall": 0.9902912621359223,
    "f1": 0.9807692307692307,
    "tp": 102,
    "fp": 3,
    "fn": 1
  },
  "table row": {
    "precision": 0.978494623655914,
    "recall": 1.0,
    "f1"

In [55]:
summary = results["evaluation"]["summary"]
print(f"mAP@0.50: {summary['mAP@0.50']:.6f}")
print(f"mAP@0.50:0.95: {summary['mAP@0.50:0.95']:.6f}")
print(f"mAP@0.75: {summary['mAP@0.75']:.6f}")
print(f"mAR@1: {summary['mAR@1']:.6f}")
print(f"mAR@10: {summary['mAR@10']:.6f}")
print(f"mAR@100: {summary['mAR@100']:.6f}")
print(f"Precision: {summary['precision']:.6f}")
print(f"Recall: {summary['recall']:.6f}")
print(f"F1: {summary['f1']:.6f}")

results["evaluation"]["class_metrics"][[
    "class_name",
    "precision",
    "recall",
    "f1",
    "tp",
    "fp",
    "fn",
    "AP@0.50",
    "mAP@0.50:0.95",
]]

mAP@0.50: 0.716095
mAP@0.50:0.95: 0.532431
mAP@0.75: 0.593518
mAR@1: 0.299870
mAR@10: 0.549534
mAR@100: 0.605447
Precision: 0.957494
Recall: 0.922414
F1: 0.939627


,class_name,precision,recall,f1,tp,fp,fn,AP@0.50,mAP@0.50:0.95
0,table,0.954545,1.000000,0.976744,21,1,0,0.993249,0.689182
1,table column,0.971429,0.990291,0.980769,102,3,1,0.985026,0.898515
2,table row,0.978495,1.000000,0.989130,273,6,0,0.991382,0.679514
3,table column header,0.863636,0.950000,0.904762,19,3,1,0.872127,0.601875
4,table projected row header,0.000000,0.000000,0.000000,0,3,22,0.000000,0.000000
5,table spanning cell,0.812500,0.520000,0.634146,13,3,12,0.454785,0.325502
